# Session 1 Playground — MAPID Custom Polygons

Fetch polygon data from MAPID API (or cached sample), explore, analyze, and visualize.

**Tracks:**
- **Core** — all mentees
- **Extend** — mid mentees (marked below)

## Setup

In [ ]:
import json
import os
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import requests

# --- Configuration ---
USE_CACHED = True  # Set False when live MAPID API is ready
LAYER_ID = "your-layer-id"
BASE_URL = os.getenv("MAPID_BASE_URL", "https://api.mapid.example/v1")
TOKEN = os.getenv("MAPID_API_TOKEN", "")
TARGET_CRS = "EPSG:32748"  # UTM 48S — adjust for your study area

DATA_DIR = Path("data")
SAMPLE_FILE = DATA_DIR / "sample_response.json"

## 1. Fetch data from MAPID API (Core)

Cells 1–4: API → GeoDataFrame

In [ ]:
def fetch_from_api(base_url: str, layer_id: str, token: str, limit: int = 50) -> dict:
    """Fetch polygon features from MAPID API (single page)."""
    url = f"{base_url}/layers/{layer_id}/features"
    headers = {"Authorization": f"Bearer {token}", "Accept": "application/json"}
    params = {"limit": limit, "offset": 0}

    resp = requests.get(url, headers=headers, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()


def load_cached(path: Path) -> dict:
    with path.open(encoding="utf-8") as f:
        return json.load(f)


if USE_CACHED:
    print(f"Using cached sample: {SAMPLE_FILE}")
    data = load_cached(SAMPLE_FILE)
else:
    print(f"Fetching from API: {BASE_URL}")
    data = fetch_from_api(BASE_URL, LAYER_ID, TOKEN)

print(f"Response keys: {list(data.keys())}")
print(f"Total features: {len(data.get('features', []))}")

In [ ]:
# Convert GeoJSON features to GeoDataFrame
features = data["features"]
gdf = gpd.GeoDataFrame.from_features(features, crs="EPSG:4326")

print(f"Rows: {len(gdf)}")
print(f"CRS: {gdf.crs}")
print(f"Geometry types: {gdf.geometry.geom_type.unique()}")
gdf.head()

### Extend: Preview pagination (mid mentees)

Skip if time is short — covered fully in Session 2.

In [ ]:
# EXTEND: Paginated fetch sketch
def fetch_all_features(base_url, layer_id, token, page_size=100):
    headers = {"Authorization": f"Bearer {token}"}
    all_features = []
    offset = 0
    while True:
        resp = requests.get(
            f"{base_url}/layers/{layer_id}/features",
            headers=headers,
            params={"limit": page_size, "offset": offset},
            timeout=30,
        )
        resp.raise_for_status()
        batch = resp.json()["features"]
        if not batch:
            break
        all_features.extend(batch)
        offset += page_size
    return all_features

# Uncomment when live API is ready:
# all_features = fetch_all_features(BASE_URL, LAYER_ID, TOKEN)
# print(f"Fetched {len(all_features)} features total")

## 2. Processing — CRS, area, validity & QA (Core)

Sample has **25 polygons**: 8 valid + 10 self-intersection/sliver (6 severe + 4 mild) + 7 overlap review.

Upload note: raw file → ~19 on MAPID; use `data/sample_mapid_upload.geojson` for all 25.

In [ ]:
# Geometry validity + QA flags from MAPID properties
gdf["geometry_valid"] = gdf.geometry.is_valid

print(f"Geometry valid: {gdf['geometry_valid'].sum()} / {len(gdf)}")
print(f"Valid baseline (qa_flag=none): {(gdf['qa_flag'] == 'none').sum()}")
print(f"Self-intersection/sliver (fixable): {gdf['fixable'].sum()}")
print(f"  severe: {(gdf['invalid_severity'] == 'severe').sum()}")
print(f"  mild: {(gdf['invalid_severity'] == 'mild').sum()}")
print(f"Overlap review: {gdf['qa_flag'].str.contains('overlap', na=False).sum()}")
print(f"Expected MAPID upload (raw): {gdf['expected_mapid_upload'].sum()} polygons")

print("\nInvalid geometry IDs:")
print(gdf.loc[~gdf["geometry_valid"], ["id", "qa_flag", "invalid_severity", "expected_mapid_upload"]])

print("\nOverlap review (valid geometry, human decision needed):")
overlap_cols = ["id", "name", "qa_flag", "overlap_with", "overlap_pct", "instructor_note"]
print(gdf[gdf["qa_flag"].str.contains("overlap", na=False)][overlap_cols])

In [ ]:
# Fix ONLY auto-fixable geometries (buffer(0) does not resolve overlap QA issues)
invalid_before = (~gdf["geometry_valid"]).sum()
fixable_before = gdf["fixable"].sum()

fix_mask = gdf["fixable"] == True
gdf.loc[fix_mask, "geometry"] = gdf.loc[fix_mask, "geometry"].buffer(0)
gdf["geometry_valid"] = gdf.geometry.is_valid

print(f"Invalid before fix: {invalid_before}")
print(f"Fixed with buffer(0): {fixable_before}")
print(f"Valid after fix: {gdf['geometry_valid'].sum()} / {len(gdf)}")
print(f"Still invalid: {gdf.loc[~gdf['geometry_valid'], 'id'].tolist()}")
print(f"Overlap rows still need manual review: {gdf['qa_flag'].str.contains('overlap', na=False).sum()}")

In [ ]:
# Compute area in projected CRS (hectares)
gdf_proj = gdf.to_crs(TARGET_CRS)
gdf["area_ha"] = gdf_proj.geometry.area / 10_000

# Centroid coordinates (WGS84 for readability)
centroids = gdf_proj.geometry.centroid.to_crs(4326)
gdf["centroid_lat"] = centroids.y
gdf["centroid_lon"] = centroids.x

gdf[["id", "name", "category", "area_ha", "geometry_valid"]].describe()

## 3. Analysis — filter and groupby (Core)

Cells 8–9

In [ ]:
# Filter polygons larger than 5 hectares
AREA_THRESHOLD_HA = 5
large = gdf[gdf["area_ha"] > AREA_THRESHOLD_HA]
print(f"Polygons > {AREA_THRESHOLD_HA} ha: {len(large)}")
large[["name", "category", "area_ha"]]

In [ ]:
# Summary by category
summary = gdf.groupby("category")["area_ha"].agg(["count", "sum", "mean"]).round(2)
summary

### Extend: Top 10 largest polygons

In [ ]:
# EXTEND: Top 10 largest
top10 = gdf.nlargest(10, "area_ha")[["name", "category", "area_ha"]]
top10

## 4. Visualization (Core)

Cells 10–11

In [ ]:
# Static choropleth — area in hectares
fig, ax = plt.subplots(figsize=(10, 8))
gdf.plot(column="area_ha", legend=True, ax=ax, cmap="YlOrRd", edgecolor="black", linewidth=0.5)
ax.set_title("MAPID Custom Polygons — Area (hectares)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

### Extend: Interactive Folium map (mid mentees)

In [ ]:
# EXTEND: Interactive map
import folium

center_lat = gdf["centroid_lat"].mean()
center_lon = gdf["centroid_lon"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=11)
folium.GeoJson(
    gdf[["name", "category", "area_ha", "geometry"]],
    tooltip=folium.GeoJsonTooltip(fields=["name", "category", "area_ha"]),
).add_to(m)
m

## Wrap-up

**Session 1 deliverable:** notebook runs top-to-bottom + choropleth screenshot.

**Next (Session 2):** Refactor this workflow into `pipeline.py` and build a mandatory n8n workflow to orchestrate 100–200 polygons.